## 1. Import Libraries

In [1]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Machine Learning (tradicional)
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import adjusted_rand_score, silhouette_score, homogeneity_score, completeness_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier

import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

## 2. Load Dataset

In [2]:
# Load training data from CSV
train_data = pd.read_csv('SVHN_train.csv')

# Separate features and labels
X_train_full = train_data.iloc[:, :-1].values.astype('float32')
y_train_full = train_data.iloc[:, -1].values


## 3. Data Preprocessing Functions

### 3.1 Shape Dataset

In [8]:
def reshape_images(X):
    """Reshape flat pixel data to 32x32x3 images using row-major interpretation"""
    return X.reshape(-1, 32, 32, 3)

def flatten_features(X):
    """Simplesmente achata as imagens para as features originais"""
    return X.reshape(X.shape[0], -1)

### 3.2 Data Selection

In [17]:
# def crop_variance_based(X, variance_threshold=3.0):
#     """
#     Remove colunas mais a direita com baixo poder discriminativo baseado na análise de variância da EDA.
#     """
    
#     # Método heurístico baseado na análise de variância
#     crop_point = max(20, min(30, int(32 - (10 - variance_threshold))))
#     X_cropped = X[:, :, :crop_point, :]
#     removed_cols = list(range(crop_point, 32))

    
#     return X_cropped, removed_cols

def manual_crop_by_pixel(X, crop_col_start=0, crop_col_end=24):
    """
    Crop manual baseado na contagem de pixels do usuário.
    """
    print(f"Aplicando crop manual a partir da coluna {crop_col_start} até {crop_col_end} inclusive.")
    

    X_cropped = X[:, :, crop_col_start:crop_col_end + 1, :]
    removed_cols = list(range(0, crop_col_start)) + list(range(crop_col_end+1, 32))

    print(f"Dimensões originais: {X.shape}")
    print(f"Dimensões após crop: {X_cropped.shape}")
    print(f"Colunas removidas: {len(removed_cols)} - {removed_cols}")
    
    return X_cropped, removed_cols

### 3.3 Standardization (Formating)

In [ ]:
def per_channel_standardization(X):
    """
    Normaliza cada canal RGB individualmente para cada imagem.
    """
    X_normalized = np.zeros_like(X, dtype=np.float32)
    
    for i in range(X.shape[0]):  # Para cada imagem
        for channel in range(3):  # Para cada canal RGB
            ch_data = X[i, :, :, channel]
            mean_ch = np.mean(ch_data)
            std_ch = np.std(ch_data)
            
            if std_ch > 1e-8:  # Evita divisão por zero
                X_normalized[i, :, :, channel] = (ch_data - mean_ch) / std_ch
            else:
                X_normalized[i, :, :, channel] = ch_data - mean_ch
    
    return X_normalized

### 3.3 Main Preprocessing

#### 3.4.1 Simple (Baseline)

In [11]:
def simple_preprocessing(X, y, test_size=0.2, sample=True, sample_size=0.2):
    """Basic preprocessing: normalize and split data"""
    # Reshape to images
    X_images = reshape_images(X)
    
    # Normalize to [0, 1]
    X_norm = X_images.astype('float32') / 255.0
    
    # Stratified split to maintain class distribution
    if sample:
        # Sample a fraction of the data for quicker experiments
        X_sample, _, y_sample, _ = train_test_split(
            X_norm, y, test_size=1 - sample_size, stratify=y, random_state=42
        )
        X_norm, y = X_sample, y_sample

    X_train, X_val, y_train, y_val = train_test_split(
        X_norm, y, test_size=test_size, stratify=y, random_state=42
    )
    
    # Compute class weights for imbalanced data
    # class_weights = compute_class_weight(
    #     'balanced', classes=np.unique(y), y=y
    # )
    # class_weight_dict = {i: weight for i, weight in enumerate(class_weights)}
    
    return X_train, X_val, y_train, y_val

#### 3.4.2 EDA Based

In [12]:
def enhanced_preprocessing_pipeline(X_train_full, y_train_full, method='variance', sample=False, sample_size=0.2):
    """
    Pipeline completo de preprocessing inteligente.
    
    Args:
        method: 'variance' ou 'manual'
        crop_param: threshold para variance ou coluna para manual
    """
    
    # 1. Reshape para imagens e normalização básica
    X_images = X_train_full.reshape(-1, 32, 32, 3).astype('float32') / 255.0
    
    # 2. Crop
    # Remove edges that do not contribute to classification or disturb it
    # if method == 'variance':
    # Right edge
    X_cropped, removed_cols = manual_crop_by_pixel(X_images, crop_col_start=6, crop_col_end=24)
    
    # Left edge
    # X_cropped, removed_cols = manual_crop_by_pixel(X_cropped, crop_col_start=6, crop_col_end=25) 
    # else:  # manual
    #     X_cropped, removed_cols = manual_crop_by_pixel(X_images, int(crop_param))
    
    # 3. Normalização per-channel
    print("\nAplicando normalização per-channel...")
    X_normalized = per_channel_standardization(X_cropped)
    
    # 4. Split estratificado
    if sample:
        # Sample a fraction of the data for quicker experiments
        X_sample, _, y_sample, _ = train_test_split(
            X_normalized, y_train_full, test_size=1 - sample_size, stratify=y_train_full, random_state=42
        )
        X_normalized, y_train_full = X_sample, y_sample
    X_train, X_val, y_train, y_val = train_test_split(
        X_normalized, y_train_full, test_size=0.2, stratify=y_train_full, random_state=42
    )
    
    # 5. Flatten para ML tradicional
    X_train_flat = X_train.reshape(X_train.shape[0], -1)
    X_val_flat = X_val.reshape(X_val.shape[0], -1)
    
    # 6. Scaling final
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_flat)
    X_val_scaled = scaler.transform(X_val_flat)
    
    # 7. Calcular class weights
    # class_weights = compute_class_weight(
    #     'balanced', classes=np.unique(y_train_full), y=y_train_full
    # )
    # class_weight_dict = {i: weight for i, weight in enumerate(class_weights)}
    
    print(f"\nDimensões finais:")
    print(f"Training: {X_train_scaled.shape}")
    print(f"Validation: {X_val_scaled.shape}")
    print(f"Redução de features: {((3072 - X_train_scaled.shape[1]) / 3072 * 100):.1f}%")
    
    return X_train_scaled, X_val_scaled, y_train, y_val, scaler, removed_cols #, class_weight_dict


## 4. Experiment Methods to Eval Data Processing

In [13]:
# classifiers funciontion
def create_classifier_models():
    """Create various classifier models for comparison"""
    models = {
        #'KNN': KNeighborsClassifier(n_neighbors=10),
        #'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42),
        #'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
        'Neural Network': MLPClassifier(hidden_layer_sizes=(1024, 256), random_state=42, max_iter=500)
    }
    return models

def train_and_evaluate_classifiers(X_train, y_train, X_val, y_val, class_weights=None):
    """Train multiple classifiers and return results"""
    
    models = create_classifier_models()
    results = {}
    
    for name, model in models.items():
        print(f"Training {name}...")
        
        # Handle class weights for models that support it
        if hasattr(model, 'class_weight') and class_weights is not None:
            model.set_params(class_weight=class_weights)
        
        # Train model
        model.fit(X_train, y_train)
        
        # Evaluate
        train_acc = model.score(X_train, y_train)
        val_acc = model.score(X_val, y_val)
        
        results[name] = {
            'model': model,
            'train_acc': train_acc,
            'val_acc': val_acc
        }
        
        print(f"{name} - Train: {train_acc:.4f}, Val: {val_acc:.4f}")
    
    return results

## 5. Exp. Visual Analysis Function

In [14]:

def plot_classifier_comparison(results, title="Classifier Comparison"):
    """Plot classifier performance comparison"""
    
    models = list(results.keys())
    train_accs = [results[model]['train_acc'] for model in models]
    val_accs = [results[model]['val_acc'] for model in models]
    
    x = np.arange(len(models))
    width = 0.35
    
    fig, ax = plt.subplots(figsize=(12, 6))
    bars1 = ax.bar(x - width/2, train_accs, width, label='Training Accuracy', alpha=0.8)
    bars2 = ax.bar(x + width/2, val_accs, width, label='Validation Accuracy', alpha=0.8)
    
    ax.set_xlabel('Models')
    ax.set_ylabel('Accuracy')
    ax.set_title(title)
    ax.set_xticks(x)
    ax.set_xticklabels(models, rotation=45)
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Add value labels on bars
    for bars in [bars1, bars2]:
        for bar in bars:
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height + 0.005,
                   f'{height:.3f}', ha='center', va='bottom', fontsize=9)
    
    plt.tight_layout()
    plt.show()

## 6. Experiment 1 (Baseline)
MLP Classifier with Flattened Features

In [ ]:
print("=== SIMPLE PREPROCESSING (BASELINE) ===")

# Apply simple preprocessing
X_train, X_val, y_train, y_val = simple_preprocessing(
    X_train_full, y_train_full, sample=False
)

print(f"Training set shape: {X_train.shape}")
print(f"Validation set shape: {X_val.shape}")
# print(f"Class weights: {class_weights}")

# Extract features (flatten pixels)
print("\nExtracting features (flattened pixels)...")
X_train_features = flatten_features(X_train)
X_val_features = flatten_features(X_val)

print(f"Training features shape: {X_train_features.shape}")
print(f"Validation features shape: {X_val_features.shape}")

# Apply scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_features)
X_val_scaled = scaler.transform(X_val_features)

# Train classifiers
print("\nTraining classifiers...")
classifier_results_baseline = train_and_evaluate_classifiers(
    X_train_scaled, y_train, X_val_scaled, y_val, class_weights=None
)

=== SIMPLE PREPROCESSING (BASELINE) ===
Training set shape: (29786, 32, 32, 3)
Validation set shape: (7447, 32, 32, 3)

Extracting features (flattened pixels)...
Training features shape: (29786, 3072)
Validation features shape: (7447, 3072)
Training set shape: (29786, 32, 32, 3)
Validation set shape: (7447, 32, 32, 3)

Extracting features (flattened pixels)...
Training features shape: (29786, 3072)
Validation features shape: (7447, 3072)

Training classifiers...
Training Neural Network...

Training classifiers...
Training Neural Network...
Neural Network - Train: 0.9580, Val: 0.7959
Neural Network - Train: 0.9580, Val: 0.7959


## 7. Experiment 2 (EDA Based)


In [18]:
print("=== CROP BASEADO NA ANÁLISE DE VARIÂNCIA ===\n")

# Testar método baseado na variância (threshold 3.0)
X_train_var, X_val_var, y_train_var, y_val_var, scaler_var, removed_cols_var = enhanced_preprocessing_pipeline(
    X_train_full, y_train_full, method='variance', sample=False
)

# Treinar MLP otimizado
print("\nTreinando MLP com features reduzidas...")
mlp_variance = MLPClassifier(
    hidden_layer_sizes=(1024, 256),
    random_state=42,
    max_iter=500
)

mlp_variance.fit(X_train_var, y_train_var)

# Avaliar
train_acc_var = mlp_variance.score(X_train_var, y_train_var)
val_acc_var = mlp_variance.score(X_val_var, y_val_var)

print(f"\nRESULTADOS:")
print(f"Training Accuracy: {train_acc_var:.4f} ({train_acc_var*100:.2f}%)")
print(f"Validation Accuracy: {val_acc_var:.4f} ({val_acc_var*100:.2f}%)")
print(f"Overfitting Gap: {(train_acc_var - val_acc_var)*100:.2f}%")
print(f"Features removidas: {len(removed_cols_var)} colunas {removed_cols_var}")

=== CROP BASEADO NA ANÁLISE DE VARIÂNCIA ===

Aplicando crop manual a partir da coluna 6 até 24 inclusive.
Dimensões originais: (37233, 32, 32, 3)
Dimensões após crop: (37233, 32, 19, 3)
Colunas removidas: 13 - [0, 1, 2, 3, 4, 5, 25, 26, 27, 28, 29, 30, 31]

Aplicando normalização per-channel...
Aplicando crop manual a partir da coluna 6 até 24 inclusive.
Dimensões originais: (37233, 32, 32, 3)
Dimensões após crop: (37233, 32, 19, 3)
Colunas removidas: 13 - [0, 1, 2, 3, 4, 5, 25, 26, 27, 28, 29, 30, 31]

Aplicando normalização per-channel...

Dimensões finais:
Training: (29786, 1824)
Validation: (7447, 1824)
Redução de features: 40.6%

Treinando MLP com features reduzidas...

Dimensões finais:
Training: (29786, 1824)
Validation: (7447, 1824)
Redução de features: 40.6%

Treinando MLP com features reduzidas...

RESULTADOS:
Training Accuracy: 0.9836 (98.36%)
Validation Accuracy: 0.8151 (81.51%)
Overfitting Gap: 16.85%
Features removidas: 13 colunas [0, 1, 2, 3, 4, 5, 25, 26, 27, 28, 29, 3